# Hafta 1 · Yazılımcı Gözüyle Kuantum Hesaplama
**Ders:** Kuantum Hesaplama ve Uygulamaları (Bilgisayar Mühendisliği Yüksek Lisans)
**Lab süresi:** ~50 dk (3. ders saati) · **Ortam:** Google Colab (GPU gerekmez)

### Bu notebook'ta neler yapacağız?
| Bölüm | Konu | Süre |
|---|---|---|
| 0 | Kurulum | 3 dk |
| A | Bit ve kübit: kendi `Qubit` sınıfımızı yazıyoruz (sadece NumPy) | 10 dk |
| B | Ölçüm = olasılıklı okuma; shot sayısı ve istatistik | 8 dk |
| C | Neden kuantum bilgisayarı klasik bilgisayarda simüle etmek zor? (bellek hesabı) | 6 dk |
| D | Qiskit ile ilk devreler: "Hello Quantum" | 10 dk |
| E | Uygulama: Kuantum rastgele sayı üreteci (QRNG) | 6 dk |
| F | Önizleme: iki kübitli Bell devresi | 3 dk |
| G | Alıştırmalar | ödev |

> **Not:** Bu derste hiç fizik kullanmıyoruz. Kübit bizim için **iki sayı tutan bir veri yapısı**, kapı **bir matris fonksiyonu**, ölçüm ise **olasılıklı bir okuma işlemi**dir.

## 0 · Kurulum
Colab'da her oturum başında bir kez çalıştırın (~30 sn). `pylatexenc`, devrelerin güzel çizilmesi için gereklidir.

In [ ]:
!pip install -q qiskit qiskit-aer pylatexenc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import qiskit, qiskit_aer
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

print("Qiskit sürümü   :", qiskit.__version__)
print("Qiskit Aer sürümü:", qiskit_aer.__version__)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, ORANGE, GRAY = "#1F3A5F", "#2E6DB4", "#D9822B", "#8A94A6"
rng = np.random.default_rng(seed=2026)   # tekrarlanabilir sonuçlar için sabit tohum

### Yardımcı fonksiyon: Bloch küresi çizimi
Bloch küresi, **tek bir kübitin durumunu 3 boyutlu bir ok olarak gösteren görselleştirme aracıdır**. Ders boyunca bu fonksiyonu kullanacağız. Şimdilik içini anlamanız gerekmiyor, sadece çalıştırın.

- Ok **yukarıyı** gösteriyorsa kübit kesin olarak `0` okunur.
- Ok **aşağıyı** gösteriyorsa kübit kesin olarak `1` okunur.
- Ok **ekvatordaysa** kübit %50–%50 okunur.

In [ ]:
def state_to_bloch(amps):
    """[alpha, beta] genlik vektörünü Bloch küresindeki (x, y, z) noktasına çevirir."""
    a, b = complex(amps[0]), complex(amps[1])
    n = np.sqrt(abs(a)**2 + abs(b)**2); a, b = a/n, b/n
    return np.array([2*(np.conj(a)*b).real, 2*(np.conj(a)*b).imag, abs(a)**2 - abs(b)**2])

def plot_bloch(amps_list, titles=None):
    """Bir veya daha fazla kübit durumunu yan yana Bloch küresinde çizer."""
    if not isinstance(amps_list[0], (list, tuple, np.ndarray)) or np.ndim(amps_list) == 1:
        amps_list = [amps_list]
    k = len(amps_list)
    fig = plt.figure(figsize=(3.6*k, 3.8))
    for i, amps in enumerate(amps_list):
        ax = fig.add_subplot(1, k, i+1, projection="3d")
        ax.set_box_aspect((1, 1, 1), zoom=1.3); ax.computed_zorder = False
        u, v = np.linspace(0, 2*np.pi, 50), np.linspace(0, np.pi, 25)
        ax.plot_surface(np.outer(np.cos(u), np.sin(v)), np.outer(np.sin(u), np.sin(v)),
                        np.outer(np.ones_like(u), np.cos(v)), color="#EEF2F8", alpha=0.25, linewidth=0, shade=False)
        t = np.linspace(0, 2*np.pi, 200)
        ax.plot(np.cos(t), np.sin(t), 0, color=GRAY, lw=0.8)
        ax.plot(np.cos(t), 0*t, np.sin(t), color=GRAY, lw=0.5); ax.plot(0*t, np.cos(t), np.sin(t), color=GRAY, lw=0.5)
        for d in [(1,0,0), (0,1,0), (0,0,1)]:
            ax.plot([-d[0], d[0]], [-d[1], d[1]], [-d[2], d[2]], color=GRAY, lw=0.7, ls="--")
        for p, s in [((0,0,1.22),"|0⟩ (z)"), ((0,0,-1.25),"|1⟩"), ((1.42,0,0),"|+⟩ (x)"), ((-1.32,0,0),"|−⟩"),
                     ((0,1.32,0),"|+i⟩ (y)"), ((0,-1.32,0),"|−i⟩")]:
            ax.text(*p, s, ha="center", va="center", fontsize=9.5, color=NAVY)
        x, y, z = state_to_bloch(amps)
        ax.plot([0, x], [0, y], [0, z], color=BLUE, lw=3); ax.scatter([x], [y], [z], color=BLUE, s=60, depthshade=False)
        ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_zlim(-1.1, 1.1)
        ax.view_init(elev=18, azim=30); ax.set_axis_off()
        if titles: ax.set_title(titles[i], fontsize=11, color=NAVY)
    plt.show()

r = 1/np.sqrt(2)
plot_bloch([[1, 0], [0, 1], [r, r]], ["[1, 0]", "[0, 1]", "[0.707, 0.707]"])

---
## A · Bit ve Kübit: Kendi `Qubit` sınıfımız

Klasik bir bit, `bool` tipinde tek bir değer tutar. **Kübit ise iki sayılık bir vektör tutar:**

$$q = [\alpha,\ \beta] \qquad \text{kural: } |\alpha|^2 + |\beta|^2 = 1$$

- $|\alpha|^2$ → okununca **0** gelme olasılığı
- $|\beta|^2$ → okununca **1** gelme olasılığı

| Özellik | Klasik bit | Kübit |
|---|---|---|
| Tuttuğu veri | `True` / `False` | İki (karmaşık) sayı: `[α, β]` |
| Okuma | Deterministik, değeri bozmaz | **Olasılıklı**, okuduktan sonra vektör `[1,0]` veya `[0,1]` olur |
| Kopyalama | `copy()` serbest | **Kopyalanamaz** (bilinmeyen bir kübitin kopyası çıkarılamaz) |
| Değiştirme | `b = not b` | Bir matrisle çarpma: `q ← U · q` |

Aşağıda, bu davranışı taklit eden **oyuncak** bir sınıf yazıyoruz. Qiskit'in içinde de mantık özünde budur.

In [ ]:
class ToyQubit:
    """Eğitim amaçlı, tek kübitlik mini simülatör (sadece NumPy)."""

    def __init__(self, alpha=1.0, beta=0.0):
        self.amps = np.array([alpha, beta], dtype=complex)
        norm = np.sum(np.abs(self.amps)**2)
        if not np.isclose(norm, 1.0):
            raise ValueError(f"Geçersiz kübit: |α|²+|β|² = {norm:.4f} (1 olmalı)")

    def probabilities(self):
        """[P(0), P(1)] döndürür."""
        return np.abs(self.amps)**2

    def apply(self, U):
        """Bir kapı (2x2 matris) uygular: q <- U @ q"""
        self.amps = U @ self.amps
        return self

    def measure(self):
        """Olasılıklı okuma: 0 veya 1 döner ve kübitin durumu ÇÖKER."""
        p0 = self.probabilities()[0]
        outcome = 0 if rng.random() < p0 else 1
        self.amps = np.array([1, 0], dtype=complex) if outcome == 0 else np.array([0, 1], dtype=complex)
        return outcome

    def __repr__(self):
        a, b = self.amps
        return f"ToyQubit(α={a:.3f}, β={b:.3f})  P(0)={abs(a)**2:.3f}, P(1)={abs(b)**2:.3f}"

q = ToyQubit(0.6, 0.8)
print("Ölçümden önce :", q)
print("Ölçüm sonucu  :", q.measure())
print("Ölçümden sonra:", q)
print("İkinci ölçüm  :", q.measure(), " <- artık hep aynı sonucu verir!")

**Gözlem:** İlk ölçümden sonra kübit `[1, 0]` ya da `[0, 1]` oldu. Tekrar ölçünce hep aynı değeri alırız. Yani **ölçüm yıkıcı bir okumadır**: orijinal olasılık bilgisi (0.6 ve 0.8) kaybolur.

➡️ Bu yüzden kuantum programlar **aynı devreyi binlerce kez (shot) çalıştırıp** sonuçların istatistiğine bakar.

Şimdi geçersiz bir kübit oluşturmayı deneyelim:

In [ ]:
try:
    ToyQubit(0.5, 0.5)
except ValueError as e:
    print("Hata:", e)

# Karmaşık sayılar da geçerlidir; olasılık, sayının büyüklüğünün karesidir: |a+bi|² = a² + b²
q = ToyQubit((1+1j)/2, 1/np.sqrt(2))
print(q)

### Kapı = matris fonksiyonu (ilk bakış)
Klasikteki `NOT` işleminin kuantum karşılığı **X kapısıdır**. X, α ile β'nın yerini değiştiren bir matristir:
$$X = \begin{bmatrix}0 & 1\\ 1 & 0\end{bmatrix} \qquad X\begin{bmatrix}\alpha\\ \beta\end{bmatrix} = \begin{bmatrix}\beta\\ \alpha\end{bmatrix}$$
**H (Hadamard)** kapısı ise kesin `0` durumunu %50–%50 durumuna çevirir:
$$H = \frac{1}{\sqrt2}\begin{bmatrix}1 & 1\\ 1 & -1\end{bmatrix} \qquad H\begin{bmatrix}1\\ 0\end{bmatrix} = \begin{bmatrix}0.707\\ 0.707\end{bmatrix}$$
(Kapıları 4. ve 5. haftada tek tek, Bloch küresi üzerinde ayrıntılı işleyeceğiz.)

In [ ]:
X = np.array([[0, 1], [1, 0]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)

q = ToyQubit()                 # başlangıç: [1, 0]
print("Başlangıç :", q)
q.apply(X);  print("X sonrası :", q)
q = ToyQubit().apply(H); print("H sonrası :", q)

plot_bloch([[1, 0], [0, 1], [r, r]], ["başlangıç [1,0]", "X uygulandı → [0,1]", "H uygulandı → [0.707,0.707]"])

---
## B · Ölçüm istatistiği: shot kavramı

Tek bir ölçüm bize sadece `0` ya da `1` verir. Olasılıkları **tahmin etmek** için aynı hazırlık + ölçüm işlemini N kez tekrarlarız. Her tekrara **shot** denir.

**Çözümlü örnek:** `q = [0.6, 0.8]`, 1000 shot
- P(0) = 0.6² = 0.36 → beklenen ≈ **360** kez `0`
- P(1) = 0.8² = 0.64 → beklenen ≈ **640** kez `1`

In [ ]:
def run_shots(alpha, beta, shots):
    counts = {"0": 0, "1": 0}
    for _ in range(shots):
        q = ToyQubit(alpha, beta)       # her shot'ta kübit YENİDEN hazırlanır
        counts[str(q.measure())] += 1
    return counts

counts = run_shots(0.6, 0.8, 1000)
print("Sayımlar:", counts)
plt.bar(counts.keys(), counts.values(), color=BLUE, width=0.5)
plt.axhline(360, color=NAVY, ls="--", lw=1); plt.axhline(640, color=NAVY, ls="--", lw=1)
plt.title("q = [0.6, 0.8], 1000 shot (kesikli çizgiler: beklenen değer)"); plt.ylabel("sayım"); plt.show()

### Kaç shot yeterli?
Tahminimizin hatası yaklaşık **standart hata** kadardır:
$$SE = \sqrt{\frac{p(1-p)}{N}}$$

| Shot (N) | SE (p = 0.5 için) | Yorum |
|---|---|---|
| 100 | 0.050 | ±%5 civarı sapma normal |
| 1.000 | 0.016 | ±%1.6 |
| 10.000 | 0.005 | ±%0.5 |

Hatayı **yarıya** indirmek için shot sayısını **4 katına** çıkarmak gerekir (çünkü √N). Gerçek kuantum bilgisayarda her shot zaman ve para demektir.

In [ ]:
shots_list = np.unique(np.logspace(1, 4.3, 50).astype(int))
estimates = [rng.binomial(n, 0.5) / n for n in shots_list]    # hızlı yol: binom dağılımından örnekleme
se = np.sqrt(0.25 / shots_list)

plt.figure(figsize=(8, 3.8))
plt.fill_between(shots_list, 0.5 - 2*se, 0.5 + 2*se, color="#E8EEF6", label="±2 SE bandı")
plt.semilogx(shots_list, estimates, "o-", ms=3, color=BLUE, label="tahmini P(0)")
plt.axhline(0.5, color=NAVY, ls="--", label="gerçek değer")
plt.xlabel("shot sayısı"); plt.ylabel("P(0) tahmini"); plt.ylim(0.2, 0.8); plt.legend(); plt.show()

for n in [100, 1000, 10000]:
    print(f"N = {n:>6}  ->  SE = {np.sqrt(0.25/n):.4f}")

---
## C · Neden klasik bilgisayarla simüle etmek zor?

n kübitlik bir sistemin durumu **2ⁿ adet karmaşık sayı** ile tanımlanır. Her karmaşık sayı (`complex128`) 16 bayt yer kaplar.

$$\text{Bellek} = 2^n \times 16 \text{ bayt}$$

**Çözümlü örnek:** 30 kübit → 2³⁰ × 16 = 17.179.869.184 bayt = **16 GiB** (bir dizüstünün tüm belleği!)

In [ ]:
def human(nbytes):
    for unit in ["B", "KiB", "MiB", "GiB", "TiB", "PiB", "EiB", "ZiB"]:
        if nbytes < 1024: return f"{nbytes:,.0f} {unit}"
        nbytes /= 1024
    return f"{nbytes:,.0f} YiB"

print(f"{'Kübit':>6} | {'Genlik sayısı (2^n)':>26} | {'Bellek (complex128)':>20}")
print("-"*60)
for n in [1, 2, 10, 20, 30, 40, 50, 60]:
    print(f"{n:>6} | {2**n:>26,} | {human(2**n * 16):>20}")

Gerçekten deneyelim: Qiskit Aer'in durum vektörü simülatörüyle kübit sayısı arttıkça süre nasıl değişiyor? (Colab'da ~1 dk sürer.)

In [ ]:
import time
sv_sim = AerSimulator(method="statevector")
ns, times = list(range(10, 25, 2)), []
for n in ns:
    qc = QuantumCircuit(n); qc.h(range(n)); qc.save_statevector()
    t0 = time.perf_counter(); sv_sim.run(qc).result(); times.append(time.perf_counter() - t0)
    print(f"{n} kübit: {times[-1]*1000:8.1f} ms   (durum vektörü: {human(2**n*16)})")

plt.figure(figsize=(7, 3.5)); plt.semilogy(ns, times, "o-", color=BLUE)
plt.xlabel("kübit sayısı"); plt.ylabel("süre (s, log)"); plt.title("Her +1 kübit ≈ 2x bellek ve süre"); plt.show()

---
## D · Qiskit ile ilk devreler

Bir kuantum programı **devre (circuit)** olarak yazılır:
1. `QuantumCircuit(n_kübit, n_klasik_bit)` ile boş devre oluştur (tüm kübitler `0` ile başlar)
2. Kapıları sırayla ekle (`qc.h(0)`, `qc.x(0)`, …)
3. `qc.measure(kübit, klasik_bit)` ile ölç
4. Simülatörde **N shot** çalıştır ve sayımları al

### D1 · Hello Quantum: kuantum yazı-tura

In [ ]:
qc = QuantumCircuit(1, 1)   # 1 kübit, 1 klasik bit
qc.h(0)                     # 0. kübite Hadamard: %50-%50 durum
qc.measure(0, 0)            # 0. kübiti ölç, sonucu 0. klasik bite yaz
qc.draw("mpl")

In [ ]:
sim = AerSimulator()
job = sim.run(transpile(qc, sim), shots=1000)
counts = job.result().get_counts()
print(counts)
plot_histogram(counts, title="H + ölçüm, 1000 shot")

### D2 · X kapısı: kuantum NOT

In [ ]:
qc_x = QuantumCircuit(1, 1)
qc_x.x(0)
qc_x.measure(0, 0)
display(qc_x.draw("mpl"))
print(sim.run(transpile(qc_x, sim), shots=1000).result().get_counts())

### D3 · Ölçmeden içeri bakmak: `Statevector`
Simülatörde "hile" yapıp ölçüm yapmadan vektörü görebiliriz (gerçek donanımda bu **mümkün değildir**, sadece ölçüm sonuçlarını görürüz).

In [ ]:
for name, build in [("boş devre", lambda c: None), ("X", lambda c: c.x(0)), ("H", lambda c: c.h(0))]:
    c = QuantumCircuit(1); build(c)
    sv = Statevector.from_instruction(c)
    print(f"{name:10s} -> vektör = {np.round(sv.data, 3)}   olasılıklar = { {str(k): round(float(v), 3) for k, v in sv.probabilities_dict().items()} }")

plot_bloch([Statevector.from_instruction(QuantumCircuit(1)).data,
            Statevector.from_label("1").data,
            Statevector.from_label("+").data], ["boş devre", "X", "H"])

### D4 · Qiskit'te bit sırası (önemli!)
Qiskit, sonuç stringlerini **sağdan sola** yazar: `'01'` sonucu → **q₀ = 1, q₁ = 0** demektir (little-endian). Bu, ilk haftalarda en sık yapılan hatadır.

In [ ]:
qc_order = QuantumCircuit(2, 2)
qc_order.x(0)                        # sadece q0'ı 1 yapıyoruz
qc_order.measure([0, 1], [0, 1])
display(qc_order.draw("mpl"))
print(sim.run(transpile(qc_order, sim), shots=100).result().get_counts(), " <- '01' = q1 q0")

---
## E · Uygulama: Kuantum Rastgele Sayı Üreteci (QRNG)

Klasik bilgisayarlardaki `random()` aslında **sözde (pseudo) rastgeledir**: aynı tohumla aynı diziyi üretir. H kapısından sonra ölçülen bir kübit ise (ideal donanımda) **gerçek** rastgeleliğin kaynağıdır.

**Fikir:** 8 kübite H uygula ve ölç → 8 rastgele bit = 1 rastgele bayt (0–255).

In [ ]:
qrng = QuantumCircuit(8, 8)
qrng.h(range(8))
qrng.measure(range(8), range(8))
qrng.draw("mpl", fold=-1)

In [ ]:
def quantum_random_bytes(n):
    """n adet rastgele bayt üretir. memory=True her shot'ın sonucunu ayrı ayrı verir."""
    res = sim.run(transpile(qrng, sim), shots=n, memory=True).result()
    return [int(bits, 2) for bits in res.get_memory()]

data = quantum_random_bytes(5000)
print("İlk 10 sayı:", data[:10])
print(f"Ortalama: {np.mean(data):.1f}  (ideal: 127.5)")

plt.figure(figsize=(8, 3.3))
plt.hist(data, bins=32, color=BLUE, edgecolor="white")
plt.title("5000 kuantum rastgele bayt: düzgün dağılım"); plt.xlabel("değer (0-255)"); plt.ylabel("sıklık"); plt.show()

> ⚠️ Burada **simülatör** kullandığımız için sonuçlar aslında yine klasik sözde rastgele sayılarla üretiliyor. Gerçek rastgelelik için devrenin gerçek bir QPU'da çalışması gerekir (7. hafta).

---
## F · Önizleme: Bell devresi (6. haftada ayrıntılı)
İki kübitli bu devrede her kübit tek başına %50–%50 rastgele, ama **iki sonuç her zaman aynı** çıkar: sadece `00` ve `11` görürüz. Bu "bağlı değişkenler" davranışına **dolanıklık (entanglement)** denir.

In [ ]:
bell = QuantumCircuit(2, 2)
bell.h(0)
bell.cx(0, 1)          # CNOT: q0 kontrol, q1 hedef
bell.measure([0, 1], [0, 1])
display(bell.draw("mpl"))
plot_histogram(sim.run(transpile(bell, sim), shots=1000).result().get_counts(), title="Bell devresi")

---
## G · Alıştırmalar
Her alıştırmada `# TODO` yazan yerleri doldurun. Altındaki `assert` satırları hata vermezse çözümünüz doğrudur.

### Alıştırma 1 · Geçerli kübit kontrolü
`is_valid_qubit(amps)` fonksiyonunu yazın: `|α|² + |β|² = 1` ise `True` döndürsün (kayan nokta hatası için `np.isclose` kullanın).

In [ ]:
def is_valid_qubit(amps):
    # TODO
    pass

assert is_valid_qubit([0.6, 0.8]) == True
assert is_valid_qubit([0.5, 0.5]) == False
assert is_valid_qubit([(1+1j)/2, 1/np.sqrt(2)]) == True
assert is_valid_qubit([1j, 0]) == True
print("Alıştırma 1: tüm testler geçti ✓")

### Alıştırma 2 · Normalizasyon
`normalize(amps)` fonksiyonu, herhangi bir sıfır olmayan vektörü geçerli bir kübit vektörüne çevirsin (vektörü uzunluğuna bölün).

In [ ]:
def normalize(amps):
    # TODO
    pass

assert np.allclose(normalize([1, 1]), [1/np.sqrt(2), 1/np.sqrt(2)])
assert np.allclose(normalize([3, 4]), [0.6, 0.8])
print("Alıştırma 2: tüm testler geçti ✓")

### Alıştırma 3 · Beklenen sayımlar
`expected_counts(amps, shots)` fonksiyonu `{'0': ..., '1': ...}` sözlüğü döndürsün. Sonra `q = [√3/2, 1/2]` için 2000 shot'lık beklenen değeri hesaplayın ve `run_shots` ile yapılan simülasyonla karşılaştırın.

In [ ]:
def expected_counts(amps, shots):
    # TODO
    pass

amps = [np.sqrt(3)/2, 1/2]
exp = expected_counts(amps, 2000)
assert np.isclose(exp["0"], 1500) and np.isclose(exp["1"], 500)
print("Beklenen :", exp)
print("Simülasyon:", run_shots(*amps, 2000))

### Alıştırma 4 · X sonra H
Önce **X** sonra **H** uygulayan tek kübitlik devreyi kurun, 1000 shot çalıştırın.
1. Histogram, sadece H uygulanan devreden farklı mı?
2. `Statevector` ile iki devrenin vektörlerini karşılaştırın. Ne fark ediyorsunuz?

In [ ]:
qc4 = QuantumCircuit(1, 1)
# TODO: X ve H kapılarını ekleyin, sonra ölçün

display(qc4.draw("mpl"))
# TODO: çalıştırın ve sayımları yazdırın

# TODO: ölçümsüz bir kopya ile Statevector'ü yazdırın

### Alıştırma 5 · Kuantum zar
`quantum_random_bytes` fikrini kullanarak **1–6 arası adil bir zar** yazın. İpucu: 3 kübit 0–7 arası sayı üretir; 6 ve 7 gelirse **yeniden atın** (rejection sampling). 600 atışın histogramını çizin.

In [ ]:
def quantum_dice(n_rolls):
    # TODO
    pass

rolls = quantum_dice(600)
assert len(rolls) == 600 and set(rolls) <= {1, 2, 3, 4, 5, 6}
plt.hist(rolls, bins=np.arange(0.5, 7.5, 1), color=BLUE, edgecolor="white"); plt.title("Kuantum zar, 600 atış"); plt.show()

### Alıştırma 6 · Bellek sınırı
`max_qubits(ram_bytes, bytes_per_amp)` fonksiyonunu yazın: verilen RAM'e sığacak en büyük kübit sayısını döndürsün.
- 16 GiB RAM, `complex128` (16 bayt) → ?
- 16 GiB RAM, `complex64` (8 bayt) → ?
- 1 TiB RAM, `complex128` → ?

In [ ]:
def max_qubits(ram_bytes, bytes_per_amp=16):
    # TODO
    pass

GiB = 2**30
print(max_qubits(16*GiB, 16), max_qubits(16*GiB, 8), max_qubits(1024*GiB, 16))
assert max_qubits(16*GiB, 16) == 30

### Alıştırma 7 · Üç kübite H (düşünme + doğrulama)
Üç kübitin her birine H uygulayıp ölçerseniz: (a) kaç farklı sonuç görürsünüz? (b) her birinin olasılığı nedir? Önce kâğıt üzerinde cevaplayın, sonra 8000 shot ile doğrulayın.

In [ ]:
qc7 = QuantumCircuit(3, 3)
# TODO

---
### Haftanın özeti
- Kübit = `[α, β]` vektörü, kural `|α|²+|β|²=1`
- Ölçüm olasılıklıdır ve durumu bozar → **shot** ile istatistik toplarız
- n kübit → 2ⁿ genlik → klasik simülasyon üstel bellek ister
- Qiskit akışı: `QuantumCircuit` → kapılar → `measure` → `transpile` → `run(shots)` → `get_counts()`
- Qiskit sonuç stringleri **sağdan sola** okunur (q₀ en sağda)

**Gelecek hafta:** Kuantum hesaplama için gereken (sadece gereken kadar!) matematik: vektörler, matrisler, karmaşık sayılar ve tensör çarpımı. NumPy ile kendi çok kübitli simülatörümüzü yazacağız.